In [2]:
import torch, torchaudio, torch.nn as nn, torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from IPython.display import Audio
Audio("obama.wav")

In [7]:
# alr at 24 kHz
mono, sr = torchaudio.load("obama.wav")
mono.shape, mono.shape[1] / sr

(torch.Size([1, 241664]), 10.069333333333333)

frames. 75 frames per second

In [8]:
latent_frame = 320 # samples
pad = - mono.shape[1] % 320
frames = F.pad(mono, (0, pad)).view(-1, latent_frame)
frames.shape, sr / latent_frame

(torch.Size([756, 320]), 75.0)

In [9]:
class Block(nn.Module):
	def __init__(self, ch, stride):
		super().__init__()
		# reduce size and expand again
		self.convs = nn.Sequential(
			nn.ELU(),
			nn.Conv1d(ch, ch // 2, kernel_size=3, padding=1),
			nn.ELU(),
			nn.Conv1d(ch // 2, ch, kernel_size=1, padding=0),
		)

		self.down = nn.Sequential(
			nn.ELU(),
			nn.Conv1d(ch, 2 * ch, kernel_size=2*stride, stride=stride, padding=(stride+1)//2)
		)
	
	def forward(self, x):
		# (B, ch, T)
		x = x + self.convs(x)
		# (B, ch, T) -> (B, 2*ch, T/stride)
		x = self.down(x)
		return x


class Encoder(nn.Module):
	def __init__(self, outdim=128):
		super().__init__()


		ch = [32, 64, 128, 256, 512]
		strides = [2, 4, 5, 8]

		self.inputconv = nn.Conv1d(1, ch[0], kernel_size=7, padding=3)
		self.blocks = nn.ModuleList([Block(c, s) for c, s in zip(ch, strides)])

		# 					  x		   h
		self.lstm = nn.LSTM(ch[-1], ch[-1], num_layers=2, batch_first=True)

		self.out = nn.Sequential(
			nn.ELU(),
			nn.Conv1d(ch[-1], outdim, kernel_size=7, padding=3)
		)

	def forward(self, wav):
		# (B, 1, T) -> (B, 32, T)
		x = self.inputconv(wav)

		for block in self.blocks:
			# (B, 32, T) -> (B, 64, T/2)
			#		-> (B, 128, T/(2*4))
			#		-> (B, 256, T/(2*4*5))
			#		-> (B, 512, T/320)
			# 		-> ( feature, timeframe ) 
			# so at that timeframe features are roughly in that frame
			x = block(x)

		# some history
		# 						         x (traverse features over time)
		# (B, 512, T/320) -> (B, T/320, 512) -> (B, 512, T/320)
		y, (h, c) = self.lstm(x.transpose(1, 2))
		x = x + y.transpose(1, 2)

		# (B, 512, T/320) -> (B, 128, T/320)
		x = self.out(x)

		return x

enc = Encoder()
encoded = enc(mono.unsqueeze(1))
encoded.shape

torch.Size([1, 128, 755])

instead of 1024^32 precision, we achieve same with 32 x 1024 by adding. each codebook gets something the other misses. 3.14159 = 3 + 0.1 + 0.04 + 0.001...

In [11]:
class Codebook(nn.Module):
	def __init__(self, K=1024, D=128, decay=0.99, eps=1e-5):
		super().__init__()
		self.decay, self.eps = decay, eps
		self.K, self.D = K, D

		# in state dict, attribute just not in optimizer.step()
		# centroids emb table (K, D) (1024, 128)
		self.register_buffer("centroids", torch.randn(K, D))

		# A temporary (K,) to add up all the assigned K embs
		self.register_buffer("cluster_size", torch.zeros(K))
		# temporary (K, D) place to sum up the embeddings passed in for averaging
		self.register_buffer("emb_sum", self.centroids.clone())
	
	def forward(self, x):
		# aka get closest embedding
		# (N, D) - (K, D)
		# (N, 1, D) - (1, K, D) = (NKD).sum(-1)
		# d = sum[ (x - emb)^2 ]
		# (N, K)
		d = torch.cdist(x, self.centroids)**2
		# (N, 1)
		k_idx = d.argmin(1)
		# (K, D) indexed N times -> (N, D) emb
		closest_k_centroids = self.centroids[k_idx]

		# update codebooks VIA K-MEANS incrementally
		if self.training:
			# (N, K) and K is onehot @ closest embedding
			onehot = F.one_hot(k_idx, self.K).type(x.dtype)
			# size = 0.99 * size + 0.01 * (K,) AKA for count # occurances in N for chosen K EMB
			self.cluster_size = self.decay * self.cluster_size + (1-self.decay) * onehot.sum(0)
			# sum = 0.99 * sum + 0.01 * (K, N) @ (N, D) AKA add all embeddings of same K group together
			self.emb_sum = self.decay * self.emb_sum + (1-self.decay) * (onehot.T @ x.detach())

			# laplace smoothing, size = (K,)
			n = self.cluster_size.sum()
			size = (self.cluster_size + self.eps) * n / (n + self.K * self.eps)

			# update k-means centroid - (K,D) / (K, 1)
			# by averaging the collected assigned vectors
			self.centroids = self.emb_sum / size.unsqueeze(1)

		# penalize loss between x and embedding
		loss = F.mse_loss(x, closest_k_centroids.detach())
		# argmin isn't differentiable. forward: emb = x + emb - x, backward: detach is treated as constant. gradient passes through
		closest_k_centroids = x + (closest_k_centroids - x).detach()

		return closest_k_centroids, k_idx, loss



class RVQ(nn.Module):
	def __init__(self, n_codebooks = 32, **kw):
		super().__init__()
		self.codebooks = nn.ModuleList([Codebook(**kw) for _ in range(n_codebooks)])

	def forward(self, x, n_books=None):
		# batch, embs, number of embs / timesteps
		B, D, T = x.shape
		# (BDT)->(BTD)->(B*T,D) aka (N, D)
		x = x.transpose(1, 2).reshape(-1, D)

		n_books = n_books or len(self.codebooks[:n_books])
		quantized_emb = 0
		toks = []
		loss = 0
		for cb in self.codebooks:
			# (N, D) -> (N,D), int, float
			k_centroids, k_idx, res_loss = cb(x)
			# (N, D) sum
			quantized_emb += k_centroids
			# (N, D) - (1,D)
			# MOST IMPORTANT. remove these centroids so the next round can get details that were missed.
			x = x - k_centroids.detach()
			# collect ids <toks> (N,1)
			toks.append(k_idx)
			# collect residual losses
			loss += res_loss

		# snapped k-centers of size (N, D) summed across all 32 codebooks
		quantized_emb = quantized_emb.reshape(B, T, D).transpose(1, 2)

		# grab the (32, N) -stack-> (N, 32) -> (B, T, 32)
		toks = torch.stack(toks, dim=1).reshape(B, T, n_books)

		return quantized_emb, toks, loss


rvq = RVQ()
approx_emb, toks, loss = rvq(encoded)
print(torch.dist(approx_emb[0], encoded[0]).item())
approx_emb.shape, toks.shape, loss.item()

551.6306762695312


(torch.Size([1, 128, 755]), torch.Size([1, 755, 32]), 82.49713134765625)

approx_emb: (B, emb, T/320). toks: (B, T/320, tok_ids)

In [ ]:
toks[0,0]

tensor([ 478,  940,  396,  456,  480,  491,  143,  522,  127,  375,  204,  985,
         395,  558,  203,  213,  338,  895,  478,  985,  704,  142,  567,  974,
         456,  782,  345,  913,  621, 1009,  846,  754])

In [29]:
class DecBlock(nn.Module):
	def __init__(self, ch, stride):
		super().__init__()
		self.up = nn.Sequential(
			nn.ELU(),
			nn.ConvTranspose1d(ch, ch // 2, kernel_size=2*stride, stride=stride, padding=(stride+1)//2, output_padding=stride%2)
		)

		ch = ch // 2
		self.convs = nn.Sequential(
			nn.ELU(),
			nn.Conv1d(ch, ch // 2, kernel_size=3, padding=1),
			nn.ELU(),
			nn.Conv1d(ch // 2, ch, kernel_size=1, padding=0),
		)
	
	def forward(self, x):
		# (B, ch, T) -> (B, ch//2, T*stride)
		x = self.up(x)
		# (B, ch//2, T*stride)
		x = x + self.convs(x)
		return x


class Decoder(nn.Module):
	def __init__(self, indim=128):
		super().__init__()


		ch = [512, 256, 128, 64, 32]
		strides = [8, 5, 4, 2]

		self.inputconv = nn.Conv1d(indim, ch[0], kernel_size=7, padding=3)

		# 					  x		   h
		self.lstm = nn.LSTM(ch[0], ch[0], num_layers=2, batch_first=True)

		self.blocks = nn.ModuleList([DecBlock(c, s) for c, s in zip(ch, strides)])
		self.out = nn.Sequential(nn.ELU(), nn.Conv1d(ch[-1], 1, kernel_size=7, padding=3))

	def forward(self, qemb):
		# (B, 128, T/320) -> (B, 512, T/320)
		x = self.inputconv(qemb)

		# undo history, recurrence
		y, (h, c) = self.lstm(x.transpose(1, 2))
		x = x + y.transpose(1, 2)

		for block in self.blocks:
			# (B, 512, T/320) -> (B, 256, T/40)
			#		-> (B, 128, T/8)
			#		-> (B, 64, T/2)
			#		-> (B, 32, T)
			x = block(x)

		# (B, 32, T) -> (B, 1, T)
		x = self.out(x)
		return x


dec = Decoder()
recovered_mono = dec(approx_emb)
recovered_mono.shape


torch.Size([1, 1, 241600])

In [28]:
from IPython.display import Audio
Audio(recovered_mono.squeeze().detach().cpu().numpy(), rate=24000)

# discriminator (training)

In [ ]:

data = [mono.unsqueeze(1)]

discr = Discriminator()

opt_g = torch.optim.Adam([*enc.parameters(), *dec.parameters()], lr=3e-4, betas=(0.5, 0.9))
opt_d = torch.optim.Adam(disc.parameters(), lr=3e-4, betas=(0.5, 0.9))

for wav in data:
	n_codebooks = random.choice([2, 4, 8, 16, 32])
	qemb, toks, l_commit = rvq(enc(wav), n_codebooks)
	wav_hat = dec(qemb)

	# --- discriminator step ---
	real_logits, _ = discr(wav)
	fake_logits, _ = discr(wav_hat)
	# updates all the discr values
	d_loss = F.BCELoss(real_logits, fake_logits)
	opt_d.zero_grad(); d_loss.backward(); opt_d.step()

	# --- generator step ---
	_, real_feats = discr(wav)
	fake_logits, fake_feats = discr(wav_hat)
	l_adv  = sum(F.relu(1 - f).mean() for f in fake_logits) # penalize the closer-to-zeros
	# penalize distance in features
	l_feat = sum(F.l1_loss(ff, rf.detach()) / rf.abs().mean() for ff, rf in zip(fake_feats, real_feats))
	l_t  = F.l1_loss(wav_hat, wav)
	l_f  = multiscale_mel_loss(wav_hat, wav)

	# updates all the enc,dec values
	g_loss = 0.1 * l_t + 1 * l_f + 3 * adv + 3 * feat + 1 * l_commit
	opt_g.zero_grad(); g_loss.backward(); opt_g.step()


	